# AI Review Intelligence — Model Exploration Notebook

This notebook walks through three approaches to predicting Yelp-style star ratings from review text:

1. **Bag-of-Words (BoW) + Logistic Regression** — the classic NLP baseline
2. **HuggingFace distilbert** — a pretrained transformer, zero-shot
3. **Claude API (AI orchestration)** — LLM reasoning layer with structured output

The goal is to show *why* AI orchestration adds value beyond a simple classifier.

In [ ]:
import os, json, re, time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from pathlib import Path

# Suppress noisy warnings
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded.')

## 1. Load Data

In [ ]:
DATA_PATH = Path('../data/sample_reviews.csv')
df = pd.read_csv(DATA_PATH)
print(f'Loaded {len(df)} reviews')
df.head()

In [ ]:
# Star rating distribution
fig, ax = plt.subplots(figsize=(6, 3))
colors = ['#6B1111', '#A32D2D', '#BA7517', '#639922', '#3B6D11']
df['stars'].value_counts().sort_index().plot(kind='bar', ax=ax, color=colors, edgecolor='none')
ax.set_xlabel('Star rating', fontsize=11)
ax.set_ylabel('Count', fontsize=11)
ax.set_title('Rating distribution in sample dataset', fontsize=12)
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig('../assets/rating_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Approach 1: Bag-of-Words + Logistic Regression

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

X = df['text']
y = df['stars']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), stop_words='english')
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec  = vectorizer.transform(X_test)

bow_model = LogisticRegression(max_iter=1000, C=1.0)
bow_model.fit(X_train_vec, y_train)

bow_preds = bow_model.predict(X_test_vec)
bow_acc   = accuracy_score(y_test, bow_preds)

print(f'BoW + LogReg accuracy: {bow_acc:.1%}')
print()
print(classification_report(y_test, bow_preds))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, bow_preds, labels=[1,2,3,4,5])
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='YlOrRd',
            xticklabels=[1,2,3,4,5], yticklabels=[1,2,3,4,5], ax=ax)
ax.set_xlabel('Predicted', fontsize=11)
ax.set_ylabel('Actual', fontsize=11)
ax.set_title('BoW + Logistic Regression — confusion matrix', fontsize=11)
plt.tight_layout()
plt.savefig('../assets/bow_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Approach 2: HuggingFace distilbert (Baseline Transformer)

In [ ]:
from transformers import pipeline

hf_pipe = pipeline(
    'text-classification',
    model='distilbert-base-uncased-finetuned-sst-2-english'
)

# HuggingFace gives binary: POSITIVE / NEGATIVE
# Map POSITIVE → 4 or 5, NEGATIVE → 1 or 2, with confidence as tiebreaker
def hf_to_stars(text: str) -> int:
    res = hf_pipe(text[:512])[0]
    label = res['label']
    score = res['score']
    if label == 'POSITIVE':
        return 5 if score > 0.9 else 4
    else:
        return 1 if score > 0.9 else 2

sample_subset = X_test.tolist()[:20]  # limit for demo
hf_preds = [hf_to_stars(t) for t in sample_subset]
hf_actual = y_test.tolist()[:20]
hf_acc = accuracy_score(hf_actual, hf_preds)

print(f'HuggingFace distilbert (mapped) accuracy on 20 samples: {hf_acc:.1%}')
print('Note: binary model mapped to star ratings — inherently limited.')

## 4. Approach 3: Claude API — AI Orchestration Layer

Instead of a classifier, we use Claude as a **reasoning engine**.
It reads the review and returns:
- Predicted stars (1–5)
- Confidence score
- Sentiment label
- Key themes (positive/negative/neutral)
- Star probability distribution
- Business advice

This is AI **orchestration** — using an LLM to produce structured output from unstructured input.

In [ ]:
import sys
sys.path.append('..')
from app.orchestrator import ReviewOrchestrator

# Set your API key here or via environment variable
API_KEY = os.environ.get('ANTHROPIC_API_KEY', 'your-key-here')
orch = ReviewOrchestrator(api_key=API_KEY)

# Analyze a few test reviews
test_reviews = X_test.tolist()[:5]
test_actual  = y_test.tolist()[:5]

claude_results = []
for i, review in enumerate(test_reviews):
    result = orch.analyze(review)
    claude_results.append(result)
    print(f'Review {i+1}: actual={test_actual[i]}, predicted={result.stars}, sentiment={result.sentiment}')
    print(f'  Themes: {[t.label for t in result.themes]}')
    print(f'  Advice: {result.advice[:80]}…')
    print()

In [ ]:
# Accuracy for the small Claude sample
claude_preds  = [r.stars for r in claude_results]
claude_acc    = accuracy_score(test_actual, claude_preds)
print(f'Claude accuracy (n=5): {claude_acc:.1%}')

# Batch summary
summary = orch.summarize_batch(claude_results)
print('\nBatch summary:')
for k, v in summary.items():
    print(f'  {k}: {v}')

## 5. Final Comparison

| Model | Approach | Granularity | Accuracy | Extra outputs |
|---|---|---|---|---|
| BoW + LogReg | Bag of words, TF-IDF | 5-class | ~60–70%* | None |
| distilbert (HuggingFace) | Pretrained transformer | Binary only | N/A for 5-class | None |
| Claude API | LLM reasoning (orchestration) | 5-class + themes + advice | ~70–80%* | Themes, confidence, business advice |

\* Accuracy varies with dataset size. The sample dataset here is intentionally tiny (5 reviews).

**Key takeaway:** Claude doesn't just predict a label — it explains *why* a review is positive or negative, identifies specific business pain points, and produces actionable advice. That's what AI orchestration adds over a simple classifier.

In [ ]:
# Visual comparison chart
models     = ['BoW + LogReg', 'distilbert\n(binary→stars)', 'Claude API\n(orchestration)']
accuracies = [0.65, 0.40, 0.75]  # illustrative — run on real data for true values
colors_bar = ['#BA7517', '#185FA5', '#3B6D11']

fig, ax = plt.subplots(figsize=(6, 3.5))
bars = ax.bar(models, accuracies, color=colors_bar, edgecolor='none', width=0.5)
for bar, val in zip(bars, accuracies):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f'{val:.0%}', ha='center', fontsize=10, color='#333')
ax.set_ylim(0, 1.0)
ax.yaxis.set_major_formatter(ticker.PercentFormatter(xmax=1))
ax.set_ylabel('5-class accuracy', fontsize=11)
ax.set_title('Model comparison — 5-class star rating prediction', fontsize=11)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('../assets/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()